# Full-res U-Net SMOKE TEST (interactive)

Proves the full-grid (192×288) pipeline **without downloading the 170 GB dataset** — everything streams from Hugging Face via range requests.

**Before running:** right panel → Accelerator → **GPU (T4 x2)**.

**What it does:** open remote full-res NetCDFs (400 days × 2 scenarios) → daily forcings → real preprocessing on the full grid (~2–3 h, runs ONCE) → 1-epoch U-Net fit (192×288, skips ON, 1 GPU) → predict → preprocessing reversal.

**Rerun-friendly:** preprocessed outputs persist in `/kaggle/working`, so a rerun skips the preprocessing and costs minutes. Stop anytime with Session Options → Stop Session — and just rerun after a fix.

## 1. Clone the repo + install deps (~2 min, skips on reruns)

In [ ]:
import os, subprocess, sys, pathlib
repo = "/kaggle/working/climx"
if os.path.isdir(repo):
    r = subprocess.run(["git", "-C", repo, "pull", "--ff-only"], capture_output=True, text=True)
    print(r.stdout or r.stderr)
else:
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/nonewanit/climx-emulator.git", repo], check=True)
os.chdir(repo)
sys.path.insert(0, str(pathlib.Path(repo) / "src"))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"], check=True)
print("setup done")

## 2. Run the smoke (live output, stage by stage)

`%run` executes in THIS process — every stage marker prints inline as it happens. The final line is `FULLRES_SMOKE_OK`.

In [ ]:
# Single-threaded dask is REQUIRED here (the remote opener shares one file
# handle; threads race on it — v5's thrash). Do NOT set CLIMX_DASK_SCHEDULER.
%run scripts/fullres_smoke.py